# SCTEC - Projeto Final — Módulo 01 | Machine Learning: Modelo Preditivo
**DESENVOLVIMENTO DE IA PARA ANÁLISE PREDITIVA**

---

## Descrição do notebook

Este material implementa um  Pipeline  Preditivo  completo aplicado à Indústria 4.0. 

●  O  algorítmo propõe uma solução preditiva para um  parque  fabril  monitorado  por  sensores, prevendo  quebras mecânicas  nos  equipamentos,  para  evitar paradas  na  linha  de  produção.  A  variável alvo  do  projeto  é  binária:  falha_maquina  =  1 (quando  há  uma  avaria  detectada)  e  falha_maquina  =  0 (funcionamento normal). 
Na solução são aplicados os conceitos de:

* **Divisão treino/teste com estratificação:** por que `stratify=y` importa quando o alvo é desbalanceado;
* **Balanceamento de classes:** SMOTE aplicado **apenas no treino**, para não vazar informação para o teste;
* **Escalonamento seletivo:** por que o KNN precisa de `StandardScaler` e a Árvore de Decisão não;
* **Ajuste de hiperparâmetros:** `n_neighbors` no KNN e `max_depth` na Árvore, e como cada um controla o overfitting;
* **Avaliação crítica de acurácia:** por que "acerto alto" pode significar pouco quando o alvo é raro.

💡 **Sobre a base:** ela foi criada com problemas propositais — nulos, duplicatas, colunas de vazamento de dados e desbalanceamento de classe — para proprorcionar as capacidades de avaliação, diagnóstico e decisão, não só sintaxe.

⚠️ **Atenção:** Trata-se de um projeto acadêmico visando ao atendimento às competências necessárias para a conclusão com aproveitamento no curso de "Desenvolvimento de IA para Análise Preditiva, do projeto SCTEC.

⚠️ **Atenção:** A cosntrução desse projeto observou a estruturação manual da hierarquia de pastas, sendo pois, necessária a sua replicação, para o seu correto funcionamento.

## FASE 0: Configuração de Ambiente e Repositório Git

### 0.1 - Ambiente e importações

Na maioria dos ambientes de desenvolvimento, as bibliotecas abaixo já está disponível. A exceção costuma ser o `imbalanced-learn` (SMOTE):
O comandos abaixo deveram ser executados no terminal, visando à instalação da biblioteca.
```
pip install imbalanced-learn
```
Para rodar localmente:
```
pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn
```

In [ ]:
# Bibliotecas de terceiros
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score
from imblearn.over_sampling import SMOTE

# Módulos da biblioteca padrão do Python
import os
import re
import json
from datetime import datetime, timedelta
from IPython.display import display

pd.set_option("display.width", 120)
sns.set_style("whitegrid")

print("Ambiente pronto!")

## FASE 1: Análise Exploratória de Dados (EDA)

<h5 style="color: #7ac1f4;">
RF00 – Carregar o Dataset de Vendas
</h5>

In [ ]:
# Carregar o CSV
df_bruto = pd.read_csv('../data/manutencao_preditiva.csv')

<h5 style="color: #7ac1f4;">
RF01 – Inspecionar e Descrever os Dados
</h5>

In [ ]:
# Função para inspecionar os dados
def inspecionar_dados(df):
    """Exibe as informações estruturais do DataFrame com tabelas renderizadas."""
    print("=== INSPEÇÃO INICIAL DO DATASET ===")
    
    # 1. Shape
    print(f'Shape: O dataframe tem [{df.shape[0]}] linhas e [{df.shape[1]}] colunas.\n')
    
    # 2. As 5 primeiras linhas (renderizadas)
    print("Primeiras 5 linhas:")
    display(df.head()) 
    
    # 3. As 5 últimas linhas (renderizadas)
    print("\nÚltimas 5 linhas:")
    display(df.tail())
    
    # 4. Tipos das colunas
    print("\nTipos das colunas:")
    display(df.dtypes.to_frame(name="Tipo de Dado"))
    
    # 5. Valores nulos
    print("\nValores nulos por coluna:")
    display(df.isnull().sum().to_frame(name="Qtd Nulos"))

In [ ]:
# Chamada da função
inspecionar_dados(df_bruto)

In [ ]:
df_bruto.info()  # Exibe informações detalhadas sobre o DataFrame
df_bruto.describe()  # Exibe estatísticas descritivas do DataFrame

<h5 style="color: #7ac1f4;">
RF02 – 3 gráficos analíticos obrigatórios
</h5>

In [ ]:
# Configuração da figura com 3 subplots em 1 linha
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# -------------------------------------------------------------------------
# GRÁFICO 1: Histograma de Distribuição por Classe (Torque em Nm)
# -------------------------------------------------------------------------
sns.histplot(
    data=df_bruto, 
    x="torque_nm", 
    hue="falha_maquina", 
    kde=True, 
    ax=axes[0], 
    palette=["#1C7293", "#D64550"]
)
axes[0].set_title("Distribuição do Torque (Nm) por Classe de Falha")
axes[0].set_xlabel("Torque (Nm)")
axes[0].set_ylabel("Frequência")

# -------------------------------------------------------------------------
# GRÁFICO 2: Gráfico de Barras do Desbalanceamento da Variável Alvo
# -------------------------------------------------------------------------
contagem = df_bruto["falha_maquina"].value_counts().sort_index()
axes[1].bar(["Sem Falha (0)", "Com Falha (1)"], contagem.values, color=["#1C7293", "#D64550"])

# Adiciona os rótulos de contagem e percentual sobre cada barra
for i, v in enumerate(contagem.values):
    percentual = (v / contagem.sum()) * 100
    axes[1].text(i, v + 150, f"{v}\n({percentual:.1f}%)", ha="center", fontsize=10, fontweight='bold')

axes[1].set_title("Desbalanceamento da Variável Alvo (falha_maquina)")
axes[1].set_ylabel("Quantidade de Equipamentos")
axes[1].set_ylim(0, max(contagem.values) * 1.15) # Ajusta limite Y para caber o texto

# -------------------------------------------------------------------------
# GRÁFICO 3: Mapa de Calor (Heatmap) da Correlação de Pearson
# -------------------------------------------------------------------------
cols_correlacao = [
    "temperatura_ar_k", "temperatura_processo_k", "velocidade_rotacao_rpm", 
    "torque_nm", "desgaste_ferramenta_min", "falha_maquina"
]

corr_matrix = df_bruto[cols_correlacao].corr(method="pearson")

sns.heatmap(
    corr_matrix, 
    annot=True, 
    fmt=".2f", 
    cmap="coolwarm", 
    center=0, 
    vmin=-1, 
    vmax=1, 
    ax=axes[2]
)
axes[2].set_title("Matriz de Correlação de Pearson")

# Ajuste fino de layout
plt.tight_layout()
plt.show()


:
### ✳️ Decisão  técnica - 01
A escolha da coluna torque_nm para o histograma da FASE 1 não foi arbitrária. Ela foi definida com base em duas análises estatísticas prévias realizadas nos itens 1 e 2 da Fase 1 (a inspeção do info() e, principalmente, a análise descritiva com o describe()).

1. Maior Dispersão e Variabilidade Estatística no describe()
Na análise descritiva dos dados numéricos:

    - Variáveis como temperatura_ar_k e temperatura_processo_k possuem amplitude e desvio padrão relativamente baixos e muito concentrados em torno da média (pouca variação relativa).
    - O torque_nm, por outro lado, apresenta uma alta amplitude e variabilidade significativa (variando desde valores próximos de 3.8 Nm até mais de 76.6 Nm), tornando-se um candidato ideal para observar como a variação de um parâmetro operacional contínuo se distribui em relação às falhas. 
    - Distribuição Normal Natural: A igualdade entre média e mediana no describe() indica um formato de curva em sino perfeito no gráfico.Picos Extremos: O max em $76.60$ isola os eventos de esforço elevado, permitindo visualizar onde ocorrem as falhas por sobrecarga nas caudas do gráfico.
    - Maior dispersão Proporcional (CV = 25%): Entre as variáveis físicas operacionais, o torque é a que possui a oscilação mais expressiva em torno da média. 

### 📊 Análise Interpretativa do Diagnóstico de EDA

A partir da inspeção das estatísticas e da visualização dos gráficos analíticos, identificou-se os seguintes padrões para o direcionamento do projeto:

1. **Distribuição do Torque e Sobrecarga Mecânica (Gráfico 1):**
   * A operação normal (Classe 0) segue uma distribuição normal centrada em aproximadamente 40 Nm. 
   * As falhas (Classe 1) concentram-se visivelmente nos extremos da distribuição, sobretudo em valores elevados de torque (> 55 Nm), indicando que episódios de sobrecarga mecânica são fortes desencadeadores de paradas não planejadas.

2. **Severo Desbalanceamento de Classes (Gráfico 2):**
   * A variável alvo `falha_maquina` possui 9.661 registros normais (96.6%) e apenas 339 falhas (3.4%).
   * *Orientação:* É fundamental aplicar reamostragem via **SMOTE** no conjunto de treino (FASE 4) para evitar viés em favor da classe majoritária.

3. **Matriz de Correlação de Pearson (Gráfico 3):**
   * **Principais Preditores:** O `torque_nm` (r = 0.19) e o `desgaste_ferramenta_min` (r = 0.11) apresentam as maiores correlações diretas com a ocorrência de falhas.
   * **Fundamentação para Feature Engineering:** A forte correlação negativa entre `velocidade_rotacao_rpm` e `torque_nm` (r = -0.88) reflete a dinâmica física dos motores elétricos, respaldando a criação do atributo de **potência mecânica** na FASE 3.
   * **Multicolinearidade Térmica:** A correlação de r = 0.88 entre `temperatura_ar_k` e `temperatura_processo_k` revela redundância de informação entre a temperatura ambiente e a do processo.

In [ ]:
count = df_bruto["id_produto"].nunique()
print(count)